# Lead scoring EDA

I explore the X Education lead data to check data quality and understand which fields may help predict conversion.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## Load the data


In [ ]:
project_root = Path.cwd()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent
data_path = project_root / "data" / "Lead Scoring.csv"
df = pd.read_csv(data_path)

# I treat blank strings and the form placeholder as missing
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.replace("Select", np.nan)

target = "Converted"
df[target] = pd.to_numeric(df[target], errors="coerce")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


In [ ]:
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1),
    "unique": df.nunique(dropna=False),
}).sort_values("missing_pct", ascending=False)
display(schema)
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print(f"Duplicate Prospect IDs: {df["Prospect ID"].duplicated().sum():,}")
print(f"Duplicate Lead Numbers: {df["Lead Number"].duplicated().sum():,}")
display(df.describe(include="all").T)


## Conversion outcome


In [ ]:
# I look at the target distribution
target_summary = df[target].value_counts(dropna=False).rename_axis(target).to_frame("leads")
target_summary["share_pct"] = (target_summary["leads"] / len(df) * 100).round(1)
display(target_summary)

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.countplot(data=df, x=target, ax=ax, color="steelblue")
ax.set(title="Lead conversion", xlabel="Converted (0 = no, 1 = yes)", ylabel="Leads")
plt.tight_layout()
plt.show()

print(f"Conversion rate: {df[target].mean():.1%}")


## Categorical fields

I treat `Prospect ID` and `Lead Number` as identifiers. The table below shows the remaining category sizes.


In [ ]:
identifier_cols = ["Prospect ID", "Lead Number"]
cat_cols = [c for c in df.select_dtypes(include="object").columns if c not in identifier_cols]
cardinality = pd.DataFrame({
    "unique_values": df[cat_cols].nunique(dropna=False),
    "missing_pct": (df[cat_cols].isna().mean() * 100).round(1),
}).sort_values("unique_values", ascending=False)
display(cardinality)


In [ ]:
# I compare common categories with conversion
candidate_cat_cols = [
    "Lead Origin", "Lead Source", "Do Not Email", "Last Activity", "Specialization",
    "What is your current occupation", "City", "A free copy of Mastering The Interview",
]
for col in candidate_cat_cols:
    summary = (
        df.groupby(col, dropna=False)[target]
        .agg(leads="size", conversion_rate="mean")
        .query("leads >= 30")
        .sort_values("conversion_rate", ascending=False)
    )
    summary["conversion_rate"] = summary["conversion_rate"].map(lambda x: f"{x:.1%}")
    print(col)
    display(summary)


In [ ]:
# I check tags and lead quality separately because they may be assigned after follow-up
for col in ["Tags", "Lead Quality", "Lead Profile", "Last Notable Activity"]:
    summary = (
        df.groupby(col, dropna=False)[target]
        .agg(leads="size", conversion_rate="mean")
        .query("leads >= 30")
        .sort_values("conversion_rate", ascending=False)
    )
    summary["conversion_rate"] = summary["conversion_rate"].map(lambda x: f"{x:.1%}")
    print(col)
    display(summary)


## Numerical fields


In [ ]:
num_cols = [
    "TotalVisits", "Total Time Spent on Website", "Page Views Per Visit",
    "Asymmetrique Activity Score", "Asymmetrique Profile Score",
]

display(df[num_cols].describe().T)

# I compare the numeric fields with conversion
for col in num_cols:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    sns.histplot(data=df, x=col, bins=30, ax=axes[0], color="steelblue")
    sns.boxplot(data=df, x=target, y=col, showfliers=False, ax=axes[1], color="lightsteelblue")
    axes[0].set_title(f"{col} distribution")
    axes[1].set_title(f"{col} by conversion")
    plt.tight_layout()
    plt.show()
    display(df.groupby(target)[col].agg(["count", "mean", "median"]).round(1))


In [ ]:
# I use Spearman correlation because these fields are skewed
corr_cols = ["TotalVisits", "Total Time Spent on Website", "Page Views Per Visit", target]
corr = df[corr_cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr, annot=True, cmap="vlag", center=0, fmt=".2f", ax=ax)
ax.set_title("Spearman correlations")
plt.tight_layout()
plt.show()


## Data quality and feature review


In [ ]:
# I check extreme values in the numeric engagement fields
outlier_rows = []
for col in num_cols:
    values = df[col].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_rows.append({
        "feature": col, "q1": q1, "q3": q3, "lower_bound": lower, "upper_bound": upper,
        "outlier_count": int(((values < lower) | (values > upper)).sum()),
    })
display(pd.DataFrame(outlier_rows).set_index("feature").round(1))


In [ ]:
# I flag identifiers, constants, and fields to check for leakage
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
feature_review = pd.DataFrame({
    "column": df.columns,
    "unique_values": [df[c].nunique(dropna=False) for c in df.columns],
    "missing_pct": [(df[c].isna().mean() * 100).round(1) for c in df.columns],
})
feature_review["review"] = "candidate; confirm available at scoring time"
feature_review.loc[feature_review["column"].isin(identifier_cols), "review"] = "identifier; exclude from model"
feature_review.loc[feature_review["column"].isin(constant_cols), "review"] = "constant; exclude from model"
feature_review.loc[feature_review["column"].isin(["Tags", "Lead Quality", "Lead Profile", "Last Activity", "Last Notable Activity"]), "review"] = "possible post-contact leakage; check timestamp and intended scoring point"
feature_review.loc[feature_review["column"] == target, "review"] = "target; never use as an input"
display(feature_review)
print("Constant columns:", constant_cols)


## EDA conclusions

- The file contains **9,240 leads and 37 columns**. There are no exact duplicate rows. `Prospect ID` and `Lead Number` are unique, so they should identify records but not be model inputs.
- `Converted` has **3,561 positives (38.5%)** and **5,679 negatives (61.5%)**. The target is imbalanced, but not extremely so.
- Blank cells and `Select` placeholders both represent missing/not-selected values. The largest gaps are `Lead Quality` (51.6%), the asymmetrique fields (45.6%), `Tags` (36.3%), profile/occupation/course preference (about 29%), and `Country` (26.6%). The modeling pipeline should preserve missingness meaningfully instead of silently dropping these rows.
- Website engagement looks promising: median `Total Time Spent on Website` is **832 seconds for converted leads versus 179 seconds for non-converted leads**. Visit counts and page views are highly skewed and have upper outliers; inspect or cap only if modeling shows they are influential.
- `Lead Origin` and `Lead Source` differ substantially in conversion rate. `Lead Add Form` and `Reference` are high-conversion groups, while `API` and `Olark Chat` are lower than average. The tiny categories need grouping or smoothing before interpreting them.
- `What is your current occupation` also separates outcomes: working professionals convert at 91.6% in this sample versus 43.6% for unemployed leads, while missing occupation is much lower. This may partly reflect how leads were collected.
- `Last Activity` and `Last Notable Activity` show strong differences, but both are time-sensitive fields. Keep them only if their values are available at the intended moment the score is generated.
- `Tags`, `Lead Quality`, and `Lead Profile` have very strong associations with conversion. The data dictionary describes tags as current lead status and quality as employee-assessed. They may encode sales follow-up or later outcomes, so I would exclude them from a prospective score unless their timestamps prove they are known before scoring.
- Several consent/ad fields are constant or nearly constant (`Magazine`, update preferences, cheque preference, and similar fields). Constant fields add no signal. `Do Not Call` and rare ad responses have too few positive examples to trust on their own.
- My initial candidate set is lead origin/source, website engagement, occupation/profile and selected form attributes. I would confirm each field’s availability time, handle missing and placeholder values deliberately, group rare categories, and validate on a held-out split before deciding final inputs.
